# Codificacion Variables Categorigas, Angulares y Temporales con metodo por Transecto y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 3 (adaptado a transectos): Codificación de variables.
Entrada: Carpetas con datos imputados:
    - imputed_by_transect/   (archivos Transecto_1.csv, Transecto_2.csv, ...)
    - imputed_global/        (archivos por estación)
Salida:
    - encoded/ml/by_transect/   (OneHot para ML, un archivo por transecto)
    - encoded/dl/by_transect/   (factorize para DL + JSON de mapping)
    - encoded/ml/global/        (OneHot para ML, un archivo por estación)
    - encoded/dl/global/        (factorize para DL + JSON de mapping)
"""

import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
INPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
INPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")

OUTPUT_BASE = os.path.join(BASE_DIR, "encoded")
OUTPUT_ML_TRANSECT = os.path.join(OUTPUT_BASE, "ml", "by_transect")
OUTPUT_DL_TRANSECT = os.path.join(OUTPUT_BASE, "dl", "by_transect")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_BASE, "ml", "global")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_BASE, "dl", "global")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Columnas que se esperan (algunas pueden no estar)
# Nota: O3 ya viene renombrada desde el código 2
NUM_COLS = ['NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.', 'Dist.', 'Angulo']
CATEGORICAL_COLS = ['Estacion', 'Transecto']   # ambas se codificarán

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def cyclical_encode(series, period):
    """Convierte una serie numérica a seno/coseno con el período dado."""
    rad = 2 * np.pi * series / period
    sin = np.sin(rad)
    cos = np.cos(rad)
    return sin, cos

def add_datetime_features(df):
    """
    A partir del índice datetime, extrae y codifica:
    - hour (0-23) -> seno/coseno (periodo 24)
    - dayofyear (1-366) -> seno/coseno (periodo 365)
    - week (1-53) -> seno/coseno (periodo 52)
    - month (1-12) -> seno/coseno (periodo 12)
    - year (entero, sin codificar)
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El índice debe ser DatetimeIndex")
    idx = df.index
    hour = idx.hour
    dayofyear = idx.dayofyear
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year

    df['hour_sin'], df['hour_cos'] = cyclical_encode(hour, 24)
    df['day_sin'], df['day_cos'] = cyclical_encode(dayofyear, 365)
    df['week_sin'], df['week_cos'] = cyclical_encode(week, 52)
    df['month_sin'], df['month_cos'] = cyclical_encode(month, 12)
    df['year'] = year
    return df

def add_directional_features(df):
    """
    Convierte Direc. y Angulo (en grados) a seno/coseno.
    Las columnas originales se eliminan.
    """
    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            rad = np.radians(df[col])
            df[f'{col}sin'] = np.sin(rad)
            df[f'{col}cos'] = np.cos(rad)
            df.drop(columns=[col], inplace=True)
    return df

# ----------------------------------------------------------------------------
# Funciones de normalización de valores categóricos
# ----------------------------------------------------------------------------

def normalize_station(val):
    """
    Convierte un valor de estación a formato 'Estacion_N'.
    Ejemplos:
        "T1_E1_Alicante" -> "Estacion_1"
        "Estacion 2"      -> "Estacion_2"
        "Estacion_3"      -> "Estacion_3"
        "E4"              -> "Estacion_4"
        "5"               -> "Estacion_5"
    """
    s = str(val).strip()
    # Buscar "Estacion" seguido de número (con espacio o guion bajo)
    m = re.search(r'Estacion[ _]?(\d+)', s, re.IGNORECASE)
    if m:
        return f"Estacion_{m.group(1)}"
    # Buscar patrón "E" seguido de número (como en T1_E1...)
    m = re.search(r'E[ _]?(\d+)', s, re.IGNORECASE)
    if m:
        return f"Estacion_{m.group(1)}"
    # Si solo es un número
    m = re.search(r'\d+', s)
    if m:
        return f"Estacion_{m.group(0)}"
    # Si no coincide, devolver el valor original sin espacios (fallback)
    return s.replace(' ', '_')

def normalize_transect(val):
    """
    Convierte un valor de transecto a formato 'Transecto_N'.
    Ejemplos:
        "Transecto 1" -> "Transecto_1"
        "T2"          -> "Transecto_2"
        "T_3"         -> "Transecto_3"
        "4"           -> "Transecto_4"
    """
    s = str(val).strip()
    # Buscar "Transecto" seguido de número
    m = re.search(r'Transecto[ _]?(\d+)', s, re.IGNORECASE)
    if m:
        return f"Transecto_{m.group(1)}"
    # Buscar "T" seguido de número (como en T1)
    m = re.search(r'T[ _]?(\d+)', s, re.IGNORECASE)
    if m:
        return f"Transecto_{m.group(1)}"
    # Si solo es un número
    m = re.search(r'\d+', s)
    if m:
        return f"Transecto_{m.group(0)}"
    # Fallback
    return s.replace(' ', '_')

def clean_column_names(col_names):
    """Reemplaza espacios y caracteres problemáticos por guiones bajos."""
    cleaned = []
    for name in col_names:
        name = str(name)
        name = name.replace(' ', '_').replace('-', '_').replace('/', '_')
        cleaned.append(name)
    return cleaned

def encode_categorical_ml(df, cat_cols):
    """
    Aplica OneHotEncoder a las columnas categóricas, previa normalización.
    Las nuevas columnas toman el nombre del valor normalizado (ej. 'Estacion_1').
    Retorna df con las dummies añadidas y sin las originales.
    """
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df

    # Normalizar valores antes de one-hot
    if 'Estacion' in existing:
        df['Estacion'] = df['Estacion'].apply(normalize_station)
    if 'Transecto' in existing:
        df['Transecto'] = df['Transecto'].apply(normalize_transect)

    # Generar dummies sin prefijo (los nombres serán los valores ya normalizados)
    dummies = pd.get_dummies(df[existing], prefix='', prefix_sep='')
    # Limpiar nombres por si acaso (aunque ya deberían estar limpios)
    dummies.columns = clean_column_names(dummies.columns)
    # Concatenar y eliminar originales
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=existing, inplace=True)
    return df

def encode_categorical_dl(df, cat_cols, save_mapping=True, mapping_file=None):
    """
    Convierte las columnas categóricas a enteros (0..n-1) usando factorize.
    Si save_mapping es True, guarda un dict con el mapeo (categoría -> código) en mapping_file.
    Retorna df con las columnas reemplazadas por enteros y el diccionario de mapeo.
    """
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df, {}
    mapping = {}
    for col in existing:
        codes, uniques = pd.factorize(df[col])
        df[col] = codes
        mapping[col] = {str(k): int(v) for v, k in enumerate(uniques)}
    if save_mapping and mapping_file:
        with open(mapping_file, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, indent=2)
    return df, mapping

def process_file(input_path, output_ml_dir, output_dl_dir, is_global=False):
    """
    Procesa un archivo CSV: aplica transformaciones y guarda versiones ML y DL.
    input_path: ruta del archivo de entrada.
    output_ml_dir: carpeta donde guardar la versión ML.
    output_dl_dir: carpeta donde guardar la versión DL.
    is_global: si es True, el archivo corresponde a una estación individual (nombre de archivo = estación).
               Si es False, corresponde a un transecto (nombre de archivo = transecto).
    """
    base_name = Path(input_path).stem
    print(f"Procesando: {base_name} (global={is_global})")

    df = pd.read_csv(input_path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Añadir características temporales
    df = add_datetime_features(df)

    # Añadir características direccionales
    df = add_directional_features(df)

    # Separar copias para ML y DL
    df_ml = df.copy()
    df_dl = df.copy()

    # Codificación ML: one-hot (con normalización)
    df_ml = encode_categorical_ml(df_ml, CATEGORICAL_COLS)

    # Codificación DL: factorize (sin normalizar, se guarda mapping)
    mapping_file = os.path.join(output_dl_dir, f"{base_name}_mapping.json")
    df_dl, _ = encode_categorical_dl(df_dl, CATEGORICAL_COLS, save_mapping=True, mapping_file=mapping_file)

    # Guardar
    ml_path = os.path.join(output_ml_dir, f"{base_name}.csv")
    dl_path = os.path.join(output_dl_dir, f"{base_name}.csv")
    df_ml.to_csv(ml_path, index=True)
    df_dl.to_csv(dl_path, index=True)
    print(f"  ML guardado en {ml_path}")
    print(f"  DL guardado en {dl_path}")

# ============================================================================
# PROCESAMIENTO
# ============================================================================

def process_by_transect():
    """Procesa todos los archivos de la carpeta imputed_by_transect."""
    if not os.path.exists(INPUT_BY_TRANSECT):
        print(f"La carpeta {INPUT_BY_TRANSECT} no existe. Se omite.")
        return
    files = list(Path(INPUT_BY_TRANSECT).glob("*.csv"))
    if not files:
        print(f"No se encontraron archivos CSV en {INPUT_BY_TRANSECT}")
        return
    for f in files:
        process_file(f, OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, is_global=False)

def process_global():
    """Procesa todos los archivos de la carpeta imputed_global."""
    if not os.path.exists(INPUT_GLOBAL):
        print(f"La carpeta {INPUT_GLOBAL} no existe. Se omite.")
        return
    files = list(Path(INPUT_GLOBAL).glob("*.csv"))
    if not files:
        print(f"No se encontraron archivos CSV en {INPUT_GLOBAL}")
        return
    for f in files:
        process_file(f, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, is_global=True)

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("Iniciando codificación de variables (transectos + global)")
    print("="*60)

    process_by_transect()
    process_global()

    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_ML_TRANSECT}")
    print(f"  - {OUTPUT_DL_TRANSECT}")
    print(f"  - {OUTPUT_ML_GLOBAL}")
    print(f"  - {OUTPUT_DL_GLOBAL}")

Iniciando codificación de variables (transectos + global)
Procesando: Transecto_1 (global=False)
  ML guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/by_transect/Transecto_1.csv
  DL guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/dl/by_transect/Transecto_1.csv
Procesando: Transecto_2 (global=False)
  ML guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/by_transect/Transecto_2.csv
  DL guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/dl/by_transect/Transecto_2.csv
Procesando: T1_E1_Alicante (global=True)
  ML guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/global/T1_E1_Alicante.csv
  DL guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/dl/global/T1_E1_Alicante.csv
Procesando: T1_E2_Elda (global=True)
  ML guardado en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/encoded/ml/global/T1_E2_Elda.csv
  DL guardado en /Volumes/copia s